# STEP 3 & 4 — Per-Mode Descriptive Profiling & Stability Analysis

## Objective
Compute descriptive statistics for each metric **within each mode** to establish a behavioral "fingerprint". Diagnose mode health by analyzing window-level variance and inactivity.

## Addressing Bias & Accuracy
*   **No Normalization Yet**: We analyze *raw* magnitude to detect absolute difficulty/bias (e.g., if one mode forces 10x more combat).
*   **Stability Check**: We look for "death cascades" or "boredom streaks" (Sparsity) which indicate non-neutral calibration candidates.


## Quick Start


In [ ]:
import pandas as pd
import numpy as np
import json
import os

DATA_FILE = os.path.join('data', 'processed', 'calibration_dataset.csv')
CONFIG_FILE = os.path.join('config', 'feature_roles.json')
OUTPUT_PROFILES = os.path.join('data', 'processed', 'mode_profiles.csv')

# Load Data & Config
if not os.path.exists(DATA_FILE):
    print("DATA FILE MISSING! Run 01_integrity_check.ipynb first.")
else:
    df = pd.read_csv(DATA_FILE)

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)
    
archetypes = config['archetypes']
all_metrics = [m for metric_list in archetypes.values() for m in metric_list]

# Filter metrics to only those present in CSV (Adaptability)
available_metrics = [m for m in all_metrics if m in df.columns]
print(f"Analyzing {len(available_metrics)} metrics across {df['modeId'].nunique()} modes.")

Analyzing 13 metrics across 3 modes.


## 1. Descriptive Statistics (Fingerprinting)
We calculate Mean, Median, Variance, and Zero-Proportion (Sparsity) for every metric, grouped by Mode.

This reveals the **Natural Elicitation** of each mode.

In [ ]:
def calculate_profiles(df, metrics, group_col='modeId'):
    profiles = []
    
    for mode, group in df.groupby(group_col):
        for metric in metrics:
            data = group[metric]
            
            stats = {
                'modeId': mode,
                'metric': metric,
                'mean': data.mean(),
                'median': data.median(),
                'std': data.std(),
                'max': data.max(),
                'sparsity_pct': (data == 0).mean() * 100  # % of windows with 0 activity
            }
            profiles.append(stats)
            
    return pd.DataFrame(profiles)

df_profiles = calculate_profiles(df, available_metrics)
print("Profiles Calculated. Sample:")
print(df_profiles.head())

Profiles Calculated. Sample:
   modeId                 metric       mean     median        std         max  \
0       1             enemiesHit   3.135802   2.000000   4.567650   21.000000   
1       1             damageDone  44.855967  28.000000  54.727392  252.000000   
2       1           timeInCombat   5.663221   4.424108   5.812306   25.127217   
3       1  deathOccurredInWindow   0.055556   0.000000   0.229772    1.000000   
4       1     deathCountInWindow   0.055556   0.000000   0.229772    1.000000   

   sparsity_pct  
0     46.913580  
1     46.913580  
2     33.333333  
3     94.444444  
4     94.444444  


## 2. Stability Analysis (Mode Health)
*   **High Sparsity** (> 80%?): Suggests boredom or lack of affordances.
*   **High Death Frequency**: Suggests overload.
*   **Variance**: Should be moderate. Zero variance = broken feature or unused mechanic.

In [ ]:
print("--- Mode Health Diagnosis ---")

modes = df_profiles['modeId'].unique()

for mode in modes:
    print(f"\nAnalyzing Mode: {mode}")
    mode_stats = df_profiles[df_profiles['modeId'] == mode]
    
    # Check Death Rate
    death_metric = mode_stats[mode_stats['metric'] == 'deathCountInWindow']
    if not death_metric.empty:
        avg_deaths = death_metric.iloc[0]['mean']
        print(f"  > Avg Deaths/Window: {avg_deaths:.4f}")
        if avg_deaths > 0.5:
             print("    WARNING: High lethality detected. Potential Overload.")
             
    # Check for Dead Mechanics (100% Sparsity)
    dead_features = mode_stats[mode_stats['sparsity_pct'] > 99]['metric'].tolist()
    if dead_features:
        print(f"  > Unused Mechanics (High Sparsity): {dead_features}")
    else:
        print("  > Good Engagement: All tracked metrics show activity.")

--- Mode Health Diagnosis ---

Analyzing Mode: 1
  > Avg Deaths/Window: 0.0556
  > Good Engagement: All tracked metrics show activity.

Analyzing Mode: 2
  > Avg Deaths/Window: 0.1103
  > Good Engagement: All tracked metrics show activity.

Analyzing Mode: 3
  > Avg Deaths/Window: 0.2258
  > Good Engagement: All tracked metrics show activity.


## 3. Save Profiles
This CSV will be used in Step 6 to derive the final parameters.

In [ ]:
df_profiles.to_csv(OUTPUT_PROFILES, index=False)
print(f"Saved Mode Profiles to: {OUTPUT_PROFILES}")

Saved Mode Profiles to: data\processed\mode_profiles.csv
